Setup

In [1]:
import os, subprocess, time, re, sys, tarfile, urllib.request, warnings, glob, shutil
warnings.filterwarnings('ignore')

print("=" * 70)
print("🚀 PRISMML FORK: Setup Binaries")
print("=" * 70)

# ── Download PrismML prebuilt binaries ───────────────────────────────
BIN_DIR = "./prism_bin"
RELEASE_TAG = "prism-b8194-1179bfc"
BASE_URL = f"https://github.com/PrismML-Eng/llama.cpp/releases/download/{RELEASE_TAG}"

try:
    nvcc_out = subprocess.check_output(["nvcc", "--version"], text=True, stderr=subprocess.STDOUT)
    cuda_match = re.search(r"release (\d+\.\d+)", nvcc_out)
    cuda_ver = float(cuda_match.group(1)) if cuda_match else 12.4
    print(f"✓ Detected CUDA: {cuda_ver}")
except Exception:
    cuda_ver = 12.4
    print("⚠ CUDA detection failed, defaulting to 12.4")

cuda_slot = "12.8" if cuda_ver >= 12.6 else "12.4"
TAR_NAME = f"llama-{RELEASE_TAG}-bin-linux-cuda-{cuda_slot}-x64.tar.gz"
TAR_URL = f"{BASE_URL}/{TAR_NAME}"
TAR_PATH = f"/tmp/{TAR_NAME}"

# Clear and re-extract
if os.path.exists(BIN_DIR):
    shutil.rmtree(BIN_DIR)
os.makedirs(BIN_DIR, exist_ok=True)

print(f"⏳ Downloading: {TAR_NAME}")
try:
    urllib.request.urlretrieve(TAR_URL, TAR_PATH)
    print(f"✅ Downloaded ({os.path.getsize(TAR_PATH)/1024/1024:.1f} MB)")
except Exception as e:
    print(f"❌ CUDA download failed: {e}")
    TAR_NAME = f"llama-{RELEASE_TAG}-bin-linux-cpu-x64.tar.gz"
    urllib.request.urlretrieve(f"{BASE_URL}/{TAR_NAME}", TAR_PATH)
    print(f"✅ CPU fallback downloaded")

print("⏳ Extracting...")
with tarfile.open(TAR_PATH, "r:gz") as tar:
    tar.extractall(BIN_DIR)

# Find binaries recursively
server_candidates = glob.glob(f"{BIN_DIR}/**/llama-server", recursive=True)
if not server_candidates:
    server_candidates = glob.glob(f"./**/llama-server", recursive=True)

if server_candidates:
    LLAMA_SERVER = server_candidates[0]
    os.chmod(LLAMA_SERVER, 0o755)
    print(f"✅ Found llama-server: {LLAMA_SERVER}")
    # Save path for other cells
    with open("/tmp/llama_server_path.txt", "w") as f:
        f.write(LLAMA_SERVER)
else:
    raise SystemExit("❌ llama-server not found!")

# ── Download Cloudflared ─────────────────────────────────────────────
if not os.path.exists("cloudflared") or os.path.getsize("cloudflared") < 1000000:
    print("⏳ Downloading Cloudflared...")
    subprocess.run("""
        (wget -q --timeout=30 -O cloudflared \
         https://github.com/cloudflare/cloudflared/releases/download/2025.2.1/cloudflared-linux-amd64 2>/dev/null || \
         curl -sL --retry 2 -o cloudflared \
         https://github.com/cloudflare/cloudflared/releases/download/2025.2.1/cloudflared-linux-amd64) && \
        chmod +x cloudflared
    """, shell=True)
    print("✅ Cloudflared ready")
else:
    print("✅ Cloudflared cached")

print("\n✅ SETUP COMPLETE")

🚀 PRISMML FORK: Setup Binaries
✓ Detected CUDA: 12.8
⏳ Downloading: llama-prism-b8194-1179bfc-bin-linux-cuda-12.8-x64.tar.gz
✅ Downloaded (150.8 MB)
⏳ Extracting...
✅ Found llama-server: ./prism_bin/llama-prism-b8194-1179bfc/llama-server
⏳ Downloading Cloudflared...
✅ Cloudflared ready

✅ SETUP COMPLETE


Download Model

In [2]:
from huggingface_hub import hf_hub_download
import os

print("=" * 70)
print("📥 Downloading Model")
print("=" * 70)

# ═════════════════════════════════════════════════════════════════════
# 🎯 PICK YOUR MODEL:
# ═════════════════════════════════════════════════════════════════════

# OPTION A: Bonsai-27B (1-bit, ~3.5GB) - CURRENTLY WORKING
HF_REPO = "prism-ml/Bonsai-27B-gguf"
MODEL_FILE = "Bonsai-27B-Q1_0.gguf"
MODEL_ALIAS = "bonsai-27b-1bit"

# OPTION B: Ternary-Bonsai-27B (2-bit, ~7.2GB)
# HF_REPO = "prism-ml/Ternary-Bonsai-27B-gguf"
# MODEL_FILE = "Ternary-Bonsai-27B-Q2_0.gguf"
# MODEL_ALIAS = "ternary-bonsai-27b"

# OPTION C: Bonsai-8B (1-bit, ~1GB) - Fast test
# HF_REPO = "prism-ml/Bonsai-8B-gguf"
# MODEL_FILE = "Bonsai-8B-Q1_0.gguf"
# MODEL_ALIAS = "bonsai-8b"

# ═════════════════════════════════════════════════════════════════════

MODEL_DIR = "./models"
os.makedirs(MODEL_DIR, exist_ok=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

expected_path = os.path.join(MODEL_DIR, MODEL_FILE)

if os.path.exists(expected_path) and os.path.getsize(expected_path) > 1000000000:
    size_gb = os.path.getsize(expected_path) / 1024**3
    print(f"✅ Cached: {MODEL_FILE} ({size_gb:.1f} GB)")
    model_path = expected_path
else:
    print(f"⏳ Downloading {MODEL_FILE} from {HF_REPO}...")
    model_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=MODEL_FILE,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
        resume_download=True,
    )
    size_gb = os.path.getsize(model_path) / 1024**3
    print(f"✅ Downloaded: {MODEL_FILE} ({size_gb:.1f} GB)")

# Save paths for next cells
with open("/tmp/model_path.txt", "w") as f:
    f.write(model_path)
with open("/tmp/model_alias.txt", "w") as f:
    f.write(MODEL_ALIAS)

print(f"\n✅ Model ready: {model_path}")
print(f"✅ Alias: {MODEL_ALIAS}")

📥 Downloading Model
⏳ Downloading Bonsai-27B-Q1_0.gguf from prism-ml/Bonsai-27B-gguf...


Bonsai-27B-Q1_0.gguf:   0%|          | 0.00/3.80G [00:00<?, ?B/s]

✅ Downloaded: Bonsai-27B-Q1_0.gguf (3.5 GB)

✅ Model ready: /content/models/Bonsai-27B-Q1_0.gguf
✅ Alias: bonsai-27b-1bit


Start llama-server

In [17]:
import subprocess, time, os

print("=" * 70)
print("🚀 Starting llama-server (CODING MODE - Thinking ENABLED)")
print("=" * 70)

with open("/tmp/llama_server_path.txt", "r") as f:
    LLAMA_SERVER = f.read().strip()
with open("/tmp/model_path.txt", "r") as f:
    model_path = f.read().strip()

subprocess.run("pkill -f llama-server 2>/dev/null || true", shell=True, capture_output=True)
time.sleep(1)

server_cmd = [
    LLAMA_SERVER,
    "-m", model_path,
    "--host", "0.0.0.0",
    "--port", "11434",
    "-c", "32768",        # 32K context for large code files
    "-ngl", "99",         # Full GPU offload
    "--temp", "0.7",
    "--top-p", "0.95",
    "--top-k", "20",
    "--jinja",
    # ⬇️ NO thinking disable flag — let model reason for coding
]

print(f"→ Command: {' '.join(server_cmd)}")
print("⏳ Starting...")

server_proc = subprocess.Popen(
    server_cmd,
    stdout=open("/tmp/llama_server.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(5)

import requests
try:
    r = requests.get("http://localhost:11434/health", timeout=3)
    if r.status_code == 200:
        print("✅ Server running on http://localhost:11434")
    else:
        print(f"⚠️ Server responded with {r.status_code}")
except Exception as e:
    print(f"⚠️ Health check: {e}")

print(f"\n✅ Server PID: {server_proc.pid}")
print("   Thinking mode: ENABLED (best for coding)")
print("   Logs: /tmp/llama_server.log")

🚀 Starting llama-server (CODING MODE - Thinking ENABLED)
→ Command: ./prism_bin/llama-prism-b8194-1179bfc/llama-server -m /content/models/Bonsai-27B-Q1_0.gguf --host 0.0.0.0 --port 11434 -c 32768 -ngl 99 --temp 0.7 --top-p 0.95 --top-k 20 --jinja
⏳ Starting...
✅ Server running on http://localhost:11434

✅ Server PID: 3212
   Thinking mode: ENABLED (best for coding)
   Logs: /tmp/llama_server.log


Open Cloudflare Tunnel

In [20]:
import subprocess, time, re, os

print("=" * 70)
print("🌐 Opening Cloudflare Tunnel")
print("=" * 70)

# Kill existing tunnels
subprocess.run("pkill -f cloudflared 2>/dev/null || true", shell=True, capture_output=True)
time.sleep(1)

p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

tunnel_url = None
start = time.time()

print("⏳ Waiting for tunnel...")
while time.time() - start < 30:
    line = p.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    if "trycloudflare" in line:
        print(f"  {line.strip()[:100]}")
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group()
        break

if tunnel_url:
    # Save to file AND display
    with open("/tmp/tunnel_url.txt", "w") as f:
        f.write(tunnel_url)

    with open("/tmp/model_alias.txt", "r") as f:
        MODEL_ALIAS = f.read().strip()

    print("\n" + "🎉" * 15)
    print("✅ TUNNEL READY!")
    print("=" * 70)
    print(f"\n🔗 URL: {tunnel_url}/v1")
    print(f"\n📋 OPENCODE CONFIG:")
    print(f'   Base URL: {tunnel_url}/v1')
    print(f'   Model:    {MODEL_ALIAS}')
    print("=" * 70)
else:
    print("❌ Tunnel failed")

🌐 Opening Cloudflare Tunnel
⏳ Waiting for tunnel...
  2026-07-16T05:29:08Z INF Requesting new quick Tunnel on trycloudflare.com...
  2026-07-16T05:29:11Z INF |  https://levels-feelings-learn-runtime.trycloudflare.com                 

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
✅ TUNNEL READY!

🔗 URL: https://levels-feelings-learn-runtime.trycloudflare.com/v1

📋 OPENCODE CONFIG:
   Base URL: https://levels-feelings-learn-runtime.trycloudflare.com/v1
   Model:    bonsai-27b-1bit


Test API

In [21]:
import requests

with open("/tmp/tunnel_url.txt", "r") as f:
    tunnel_url = f.read().strip()
with open("/tmp/model_alias.txt", "r") as f:
    MODEL_ALIAS = f.read().strip()

url = f"{tunnel_url}/v1/chat/completions"

payload = {
    "model": MODEL_ALIAS,
    "messages": [
        {"role": "system", "content": "You are a helpful coding assistant."},
        {"role": "user", "content": "Write a Python function to reverse a string. Include comments."}
    ],
    # ⬇️ Large token budget for thinking + code
    "max_tokens": 1024,
    "temperature": 0.7
}

print("⏳ Testing with thinking mode ON (coding task)...")
response = requests.post(url, json=payload, timeout=120)

if response.status_code == 200:
    result = response.json()
    reply = result["choices"][0]["message"]["content"]
    print(f"\n🤖 Response:\n{'='*60}")
    print(reply)
    print(f"{'='*60}")
    print(f"\n📊 Tokens: {result.get('usage', {})}")
else:
    print(f"❌ Error {response.status_code}: {response.text[:500]}")

⏳ Testing with thinking mode ON (coding task)...

🤖 Response:
Here's a clean, Pythonic implementation with detailed comments:

```python
def reverse_string(text: str) -> str:
    """
    Reverses the given string.
    
    Args:
        text (str): The string to be reversed.
        
    Returns:
        str: A new string with the characters in reverse order.
    """
    # Python's slice notation [start:stop:step] provides an efficient way to reverse a string.
    # Omitting `start` and `stop` makes it cover the entire string.
    # Setting `step` to `-1` tells Python to iterate backwards, from the last character to the first.
    return text[::-1]


# Example usage:
if __name__ == "__main__":
    original = "Hello, Python!"
    reversed_text = reverse_string(original)
    print(f"Original: {original}")

📊 Tokens: {'completion_tokens': 1024, 'prompt_tokens': 34, 'total_tokens': 1058}


Quick Local Test (Optional)

In [22]:
import requests

url = "http://localhost:11434/v1/chat/completions"

payload = {
    "model": "bonsai-27b-1bit",
    "messages": [{"role": "user", "content": "Say 'Hello'"}],
    "max_tokens": 512,   # ← Increase from 20 to 512
}

print("⏳ Testing...")
r = requests.post(url, json=payload, timeout=120)
print(f"Status: {r.status_code}")
print(f"Response: '{r.json()['choices'][0]['message']['content']}'")

⏳ Testing...
Status: 200
Response: 'Hello'
